In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

100%|██████████| 149M/149M [00:01<00:00, 104MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1


In [ ]:
%mv /root/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1/Testing /content

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import pickle
from sklearn.model_selection import train_test_split

# ----------------------------------
# Load Pickled Keras Model
# ----------------------------------
with open('/content/ModelPretrained.pkl', 'rb') as f:
    model = pickle.load(f)

# ----------------------------------
# Prepare Generators (already done in your code)
# ----------------------------------
# Assume: df_train, df_test are already defined in notebook
valid_ts, df_test = train_test_split(df_test, test_size=0.5, random_state=42)
tr_gen = ImageDataGenerator(rescale=1/255)
ts_gen = ImageDataGenerator(rescale=1/255)

batchsize = 32
img_size = (224, 224)

gen_train = tr_gen.flow_from_dataframe(
    df_train, x_col='filepath', y_col='label',
    target_size=img_size, class_mode='categorical',
    batch_size=batchsize, shuffle=True, color_mode='rgb')

gen_valid = ts_gen.flow_from_dataframe(
    valid_ts, x_col='filepath', y_col='label',
    target_size=img_size, class_mode='categorical',
    batch_size=batchsize, shuffle=True, color_mode='rgb')

gen_test = ts_gen.flow_from_dataframe(
    df_test, x_col='filepath', y_col='label',
    target_size=img_size, class_mode='categorical',
    batch_size=1, shuffle=False, color_mode='rgb')

class_dict = gen_train.class_indices
inv_class_dict = {v: k for k, v in class_dict.items()}

# ----------------------------------
# Grad-CAM Implementation (Keras)
# ----------------------------------
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Get gradients of the predicted class wrt last conv layer output
    grads = tape.gradient(class_channel, conv_outputs)

    # Mean intensity of the gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply each channel by 'how important' it is
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize to [0, 1]
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(heatmap, original_img, alpha=0.4, colormap=cv2.COLORMAP_JET):
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), colormap)
    superimposed_img = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)
    return superimposed_img

# ----------------------------------
# Apply Grad-CAM to Test Sample
# ----------------------------------
# Choose a test image from generator
img, label = gen_test.next()
pred = model.predict(img)
pred_class = np.argmax(pred)
true_class = np.argmax(label)

# Get heatmap
last_conv_layer = [layer.name for layer in model.layers if 'conv' in layer.name][-1]  # auto-detect last conv
heatmap = get_gradcam_heatmap(model, img, last_conv_layer, pred_class)

# Convert image for display
img_disp = (img[0] * 255).astype(np.uint8)

# Overlay heatmap
result = overlay_heatmap(heatmap, img_disp)

# Show result
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title(f"True: {inv_class_dict[true_class]}, Pred: {inv_class_dict[pred_class]}")
plt.imshow(img_disp)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Grad-CAM")
plt.imshow(result)
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------
# STEP 1: Import Libraries
# ------------------------------
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import cv2
from glob import glob
import random

# ------------------------------
# STEP 2: Paths and Parameters
# ------------------------------
# Path to your saved model (choose the one you trained)
model_path = '/content/Model_Final_Brain_Tumor.h5'  # or 'Model Final Brain Tumor.h5'

# Path to test image directory
test_dir = '/content/Testing'  # replace with your actual dataset path

# Image input size
img_size = (224, 224)

# ------------------------------
# STEP 3: Load Trained Model
# ------------------------------
model = load_model(model_path)
model.summary()

# ------------------------------
# STEP 4: Grad-CAM Utilities
# ------------------------------
def get_img_array(img_path, size):
    """Load and preprocess image."""
    img = image.load_img(img_path, target_size=size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    return arr / 255.0

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap."""
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(heatmap, original_img, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Overlay heatmap on image."""
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), colormap)
    overlayed = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)
    return overlayed

# ------------------------------
# STEP 5: Run Grad-CAM on Test Image
# ------------------------------
# Randomly pick a test image
classes = os.listdir(test_dir)
chosen_class = random.choice(classes)
test_img_path = random.choice(glob(os.path.join(test_dir, chosen_class, '*.jpg')))

print(f"Using test image: {test_img_path}")

# Load and preprocess image
img_array = get_img_array(test_img_path, img_size)

# Predict
preds = model.predict(img_array)
pred_class_index = np.argmax(preds[0])
pred_class_label = list(model.class_names)[pred_class_index] if hasattr(model, 'class_names') else pred_class_index

# ------------------------------
# STEP 6: Get Last Conv Layer Name
# ------------------------------
# Automatically detect last Conv2D layer
last_conv_layer = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

if last_conv_layer is None:
    raise ValueError("No Conv2D layer found in model!")

print(f"Last Conv Layer: {last_conv_layer}")

# ------------------------------
# STEP 7: Generate and Show Grad-CAM
# ------------------------------
# Create heatmap
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer)

# Load original image
original_img = cv2.imread(test_img_path)
original_img = cv2.resize(original_img, img_size)

# Overlay heatmap
overlayed_img = overlay_heatmap(heatmap, original_img)

# Display
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM\nPrediction: {pred_class_label}")
plt.imshow(cv2.cvtColor(overlayed_img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.tight_layout()
plt.show()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)          │ (None, 7, 7, 1536)          │      10,783,535 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1536)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         393,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │           1,028 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,178,037 (42.64 MB)

 Trainable params: 11,090,732 (42.31 MB)

 Non-trainable params: 87,303 (341.03 KB)

 Optimizer params: 2 (12.00 B)

Using test image: /content/Testing/pituitary/Te-pi_0261.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


ValueError: No Conv2D layer found in model!

##-------------------------


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import cv2
from glob import glob
import random

# ------------------------------
# STEP 1: CONFIGURATION
# ------------------------------
model_path = '/content/Model_Final_Brain_Tumor.h5'  # Change if your model filename is different
test_dir = '/content/Testing'  # Update path if needed
img_size = (224, 224)

# ------------------------------
# STEP 2: LOAD MODEL
# ------------------------------
model = load_model(model_path)
print("\n✅ Model loaded successfully!\n")
model.summary()

# ------------------------------
# STEP 3: IMAGE & GRAD-CAM UTILS
# ------------------------------
def get_img_array(img_path, size):
    """Load and preprocess image."""
    img = image.load_img(img_path, target_size=size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    return arr / 255.0

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate a Grad-CAM heatmap."""
    try:
        grad_model = tf.keras.models.Model(
            [model.inputs],
            [model.get_layer(last_conv_layer_name).output, model.output]
        )
    except ValueError as e:
        raise ValueError(f"Layer '{last_conv_layer_name}' not found in model. Error: {e}")

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(heatmap, original_img, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Overlay heatmap on the original image."""
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), colormap)
    overlayed = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)
    return overlayed

def find_last_conv_layer_nested(model):
    """Find the last Conv2D layer (even inside nested models)."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
        elif hasattr(layer, 'layers'):  # nested model
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    print(f"✅ Found nested Conv2D layer: {sublayer.name}")
                    return sublayer.name
    raise ValueError("No Conv2D layer found (even nested ones).")

# ------------------------------
# STEP 4: PICK RANDOM TEST IMAGE
# ------------------------------
if not os.path.exists(test_dir):
    raise FileNotFoundError(f"❌ Test directory not found: {test_dir}")

classes = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
if not classes:
    raise ValueError("❌ No class folders found in test directory.")

chosen_class = random.choice(classes)
image_paths = glob(os.path.join(test_dir, chosen_class, '*.jpg'))

if not image_paths:
    raise FileNotFoundError(f"❌ No images found in class folder: {chosen_class}")

test_img_path = random.choice(image_paths)
print(f"\n📷 Selected test image: {test_img_path}\n")

img_array = get_img_array(test_img_path, img_size)

# ------------------------------
# STEP 5: PREDICT CLASS
# ------------------------------
pred = model.predict(img_array)
pred_class = np.argmax(pred[0])
print(f"🔮 Predicted class index: {pred_class}")

# ------------------------------
# STEP 6: FIND LAST CONV LAYER
# ------------------------------
last_conv_layer = find_last_conv_layer_nested(model)
print(f"🔥 Last Conv2D layer for Grad-CAM: {last_conv_layer}")

# ------------------------------
# STEP 7: GENERATE & OVERLAY HEATMAP
# ------------------------------
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer)

original_img = cv2.imread(test_img_path)
original_img = cv2.resize(original_img, img_size)
overlayed_img = overlay_heatmap(heatmap, original_img)

# ------------------------------
# STEP 8: DISPLAY RESULTS
# ------------------------------
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM (Predicted class: {pred_class})")
plt.imshow(cv2.cvtColor(overlayed_img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.tight_layout()
plt.show()



✅ Model loaded successfully!



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)          │ (None, 7, 7, 1536)          │      10,783,535 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1536)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         393,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │           1,028 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,178,037 (42.64 MB)

 Trainable params: 11,090,732 (42.31 MB)

 Non-trainable params: 87,303 (341.03 KB)

 Optimizer params: 2 (12.00 B)


📷 Selected test image: /content/Testing/meningioma/Te-me_0285.jpg

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
🔮 Predicted class index: 1
✅ Found nested Conv2D layer: top_conv
🔥 Last Conv2D layer for Grad-CAM: top_conv


ValueError: Layer 'top_conv' not found in model. Error: No such layer: top_conv. Existing layers are: ['efficientnetb3', 'global_average_pooling2d', 'dense', 'dropout', 'dense_1'].